# RetailIQ Data Quality and Cleaning

## tl;dr

The source contains 1,067,371 rows. Exactly 12,133 duplicate source rows are removed, while sales, cancellations, adjustments, zero-price records, and anonymous purchases remain separately auditable.

## Context & Methods

This diagnostic validates the transaction grain and documents the effect of every cleaning class before business analysis.

### Key Assumptions

Revenue is reported in the source currency. Cancellations reduce net revenue only when a positive source price is available.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
pd.options.display.float_format = '{:,.2f}'.format

## Data

### 1. Load the cleaning evidence

In [2]:
transactions = pd.read_parquet(ROOT / 'data/processed/transactions.parquet')
summary = pd.read_csv(ROOT / 'data/processed/cleaning_summary.csv')
print(f'Clean rows: {len(transactions):,}')
display(summary)

Clean rows: 1,055,238


,transaction_status,rows,gross_revenue,returned_value,net_revenue,row_share
0,adjustment,5863,0.00,0.00,0.00,0.01
1,cancellation_or_return,21681,0.00,"723,358.58","-723,358.58",0.02
2,invalid,5,0.00,0.00,0.00,0.00
3,valid_sale,1024985,"20,064,123.05",0.00,"20,064,123.05",0.96
4,zero_price,2704,0.00,0.00,0.00,0.00
5,exact_duplicate_removed,12133,NaN,NaN,NaN,0.01


## Results

### 2. Check completeness and transaction classes

In [3]:
quality = pd.DataFrame({
 'missing_count': transactions.isna().sum(),
 'missing_rate': transactions.isna().mean()
}).sort_values('missing_rate', ascending=False)
display(quality.head(10))
display(transactions['transaction_status'].value_counts().rename_axis('status').to_frame('rows'))

,missing_count,missing_rate
customer_id,242870,0.23
description,4382,0.00
invoice,0,0.00
transaction_id,0,0.00
quantity,0,0.00
stock_code,0,0.00
invoice_date,0,0.00
unit_price,0,0.00
country,0,0.00
source_sheet,0,0.00


,rows
status,
valid_sale,1024985
cancellation_or_return,21681
adjustment,5863
zero_price,2704
invalid,5


### 3. Reconcile monetary components

In [4]:
money = transactions[['gross_line_revenue','returned_value','net_line_revenue']].sum()
display(money.to_frame('amount'))
assert np.isclose(money['gross_line_revenue'] - money['returned_value'], money['net_line_revenue'])

,amount
gross_line_revenue,"20,064,123.05"
returned_value,"723,358.58"
net_line_revenue,"19,340,764.47"


## Takeaways

The retained statuses make sales, cancellations, adjustments, and unusable records auditable. Anonymous transactions remain in revenue reporting but are not eligible for customer-level modelling.